# Slurm Manager Validation

This notebook validates the logic of `scripts/slurm_manager.py` by performing dry-runs of script generation and configuration loading.

In [ ]:
import sys
import os
sys.path.append('..')

import yaml
from pathlib import Path
import datetime

## 1. Config Loading Test

In [ ]:
config_path = Path("../config/remote/slurm.yaml")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

print(f"Host: {cfg['host']}")
print(f"Remote Dir: {cfg['remote_dir']}")

## 2. Script Generation Dry-Run

In [ ]:
def dry_run_submit(cfg, exp_overrides):
    slurm_cfg = cfg['slurm_defaults']
    job_name = "test_job"
    remote_dir = cfg['remote_dir']
    
    script_content = [
        "#!/bin/bash",
        f"#SBATCH --job-name={job_name}",
        f"#SBATCH --partition={slurm_cfg['partition']}",
        f"#SBATCH --time={slurm_cfg['time']}",
        f"#SBATCH --nodes={slurm_cfg['nodes']}",
        f"#SBATCH --ntasks={slurm_cfg['ntasks']}",
        f"#SBATCH --cpus-per-task={slurm_cfg['cpus_per_task']}",
        f"#SBATCH --mem={slurm_cfg['mem']}",
        f"#SBATCH --output=logs/%x_%j.out",
        f"#SBATCH --error=logs/%x_%j.err",
        "",
        f"cd {remote_dir}",
        "mkdir -p logs results outputs",
        f"export PYTHONPATH=$PYTHONPATH:{remote_dir}",
        f"{cfg['python_env']} main.py {exp_overrides}"
    ]
    return "\n".join(script_content)

test_script = dry_run_submit(cfg, "loading.name=dummy")
print("--- Generated Script Sample ---")
print(test_script)